In [18]:
#Importe
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

### 1. Kodierung ###

In [2]:
#1.1
df = pd.read_excel("listings_adjusted_iqr_bereinigt.xlsx")

drop_columns = [
    "id",
    "listing_url",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
    "number_of_reviews"
]

df_cleaned = df.drop(columns=drop_columns)

df_cleaned["instant_bookable"] = df_cleaned["instant_bookable"].map({
    "t": 1,
    "f": 0
})

#1.3
categorical_columns = [
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "property_type",
    "room_type"
]

df = pd.get_dummies(
    df_cleaned,
    columns=categorical_columns,
    dtype=int
)


#Speichern
df.to_excel(
    "listings_adjusted_iqr_bereinigt_kodiert.xlsx",
    index=False
)

print("Neue Datei erstellt.")

Neue Datei erstellt.


# 2. Decision Trees #

Variablen inbegriffen:
•	Neighbourhood_cleansed
•	Neighbourhood_group_cleansed
•	Property_type
•	Room_type
•	Accommodates
•	Price
•	Minimum_nights
•	Availability_365
•	Instant_bookable
•	Calculated_host_listings_count
•	Is_superhost
•	Is_instant_bookable


## 2.1 mit allen ##

In [4]:
#2.1
df = pd.read_excel("listings_adjusted_iqr_bereinigt_kodiert.xlsx")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

#2.2
#2.2.1
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#2.2.2
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [5]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.23539892830548903
Test R²: 0.15851053986266217


In [6]:
#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                               Feature  Importance
5                       calculated_host_listings_count    0.318313
6                                         is_superhost    0.299016
3                                     availability_365    0.133140
1                                                price    0.063368
7                                  is_instant_bookable    0.041193
0                                         accommodates    0.034285
2                                       minimum_nights    0.030472
149                neighbourhood_group_cleansed_Pankow    0.016596
179    property_type_Private room in bed and breakfast    0.016344
147                 neighbourhood_group_cleansed_Mitte    0.009552
170                   property_type_Entire rental unit    0.008049
163                         property_type_Entire condo    0.007503
153  neighbourhood_group_cleansed_Tempelhof - Schön...    0.004235
125        neighbourhood_cleansed_Tempelhofer Vorstadt    0.00

Probieren anderer Bäume mit beschränken der max_depth und min_samples_leaf

In [26]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=16,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.2864061365027629
Test R²: 0.21647642502095288


In [27]:
#2.4
#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                              Feature  Importance
5                      calculated_host_listings_count    0.420874
6                                        is_superhost    0.416962
3                                    availability_365    0.069120
7                                 is_instant_bookable    0.050335
214                         room_type_Entire home/apt    0.018067
2                                      minimum_nights    0.016910
4                                    instant_bookable    0.005271
1                                               price    0.002462
8                    neighbourhood_cleansed_Adlershof    0.000000
0                                        accommodates    0.000000
10              neighbourhood_cleansed_Alexanderplatz    0.000000
11             neighbourhood_cleansed_Allende-Viertel    0.000000
12                neighbourhood_cleansed_Alt  Treptow    0.000000
13   neighbourhood_cleansed_Alt-Hohenschönhausen Nord    0.000000
14    neig

In [28]:
from sklearn.tree import export_text

tree_rules = export_text(
    tree,
    feature_names=list(X.columns)
)

print(tree_rules)

|--- calculated_host_listings_count <= 2.50
|   |--- is_superhost <= 0.50
|   |   |--- is_instant_bookable <= 0.50
|   |   |   |--- availability_365 <= 262.50
|   |   |   |   |--- value: [4.85]
|   |   |   |--- availability_365 >  262.50
|   |   |   |   |--- value: [4.75]
|   |   |--- is_instant_bookable >  0.50
|   |   |   |--- availability_365 <= 296.00
|   |   |   |   |--- value: [4.71]
|   |   |   |--- availability_365 >  296.00
|   |   |   |   |--- value: [4.59]
|   |--- is_superhost >  0.50
|   |   |--- instant_bookable <= 0.50
|   |   |   |--- availability_365 <= 310.50
|   |   |   |   |--- value: [4.91]
|   |   |   |--- availability_365 >  310.50
|   |   |   |   |--- value: [4.86]
|   |   |--- instant_bookable >  0.50
|   |   |   |--- minimum_nights <= 6.00
|   |   |   |   |--- value: [4.86]
|   |   |   |--- minimum_nights >  6.00
|   |   |   |   |--- value: [4.75]
|--- calculated_host_listings_count >  2.50
|   |--- is_superhost <= 0.50
|   |   |--- availability_365 <= 347.50


## 2.2 Decision Tree mit Cross-validation auf Trainingsdaten ##

In [19]:
# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

# Finaler Testdatensatz
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
from sklearn.model_selection import cross_validate, KFold
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=16,
    random_state=42
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error"
}

cv_results_tree = cross_validate(
    tree,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

print("Decision Tree Cross-Validation")
print("Mittleres CV R²:", round(cv_results_tree["test_r2"].mean(), 3))
print("SD CV R²:", round(cv_results_tree["test_r2"].std(), 3))
print("Mittleres CV MAE:", round(-cv_results_tree["test_mae"].mean(), 3))
print("Mittleres CV RMSE:", round(-cv_results_tree["test_rmse"].mean(), 3))

Decision Tree Cross-Validation
Mittleres CV R²: 0.277
SD CV R²: 0.017
Mittleres CV MAE: 0.147
Mittleres CV RMSE: 0.227


In [24]:
tree.fit(X_train, y_train)

test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)

# Erst MSE berechnen, dann Wurzel ziehen
test_mse = mean_squared_error(y_test, test_predictions)
test_rmse = np.sqrt(test_mse)

print("Finaler Test Decision Tree")
print("Test R²:", round(test_r2, 3))
print("Test MAE:", round(test_mae, 3))
print("Test RMSE:", round(test_rmse, 3))

Finaler Test Decision Tree
Test R²: 0.216
Test MAE: 0.151
Test RMSE: 0.229


# 3. Random Forest #